In [1]:
!rm -rf InkubaLM-Challenge
!git clone https://github.com/melissafasol/InkubaLM-Challenge.git
%cd InkubaLM-Challenge

Cloning into 'InkubaLM-Challenge'...
remote: Enumerating objects: 223, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 223 (delta 6), reused 7 (delta 1), pack-reused 202 (from 1)
Receiving objects: 100% (223/223), 950.75 KiB | 4.08 MiB/s, done.
Resolving deltas: 100% (133/133), done.
/content/InkubaLM-Challenge


In [2]:
!git checkout refactor-src-structure
!git pull

Branch 'refactor-src-structure' set up to track remote branch 'refactor-src-structure' from 'origin'.
Switched to a new branch 'refactor-src-structure'
Already up to date.


In [3]:
!pip install datasets
!pip install transformers datasets peft trl accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is

In [9]:
import sys
sys.path.append("..")  # Add parent directory to the path

import os
from typing import List
from pathlib import Path
import numpy as np

# DO NOT EDIT
# create submission file
import pandas as pd
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
)

from utils import (
    model_function,
    eval
    )

from src import (
    data_utils,
    model_utils,
    inference,
    prompts,
    evaluation,
    )

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset, Value, DatasetDict

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import PeftModel, PeftConfig
from sklearn.model_selection import train_test_split


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
#os.environ["TOKENIZERS_PARALLELISM"] = "false"

from huggingface_hub import login

try:
    from google.colab import userdata

    # Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
    # vars as appropriate for your system.
    # userdata.get("HF_TOKEN") indicates that the name of the token in the Colab env is HF_TOKEN
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except:
    os.environ["HF_TOKEN"] = "----"

login(token=os.environ["HF_TOKEN"])

token = os.environ["HF_TOKEN"]
if token == "----":
    print("⚠️ Warning: No Hugging Face token found. Some models may not load.")
else:
    login(token=token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Load Base Model and Run Inference on Test Set for Baseline Performance

In [10]:
model_name = "lelapa/InkubaLM-0.4B"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.environ["HF_TOKEN"])
model = model_function.load_model(model_name)

config.json:   0%|          | 0.00/763 [00:00<?, ?B/s]

vulavulaslm.py:   0%|          | 0.00/42.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/lelapa/InkubaLM-0.4B:
- vulavulaslm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [12]:
output_path = "/content/drive/MyDrive/InkubaLM-Challenge/Output/baseline_performance"
os.makedirs(output_path, exist_ok=True)

In [8]:
BASE_PROMPT = "Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n ### Instruction: {}\n\n ### Response: "
sent_instruction = "Please identify the sentiment reflected in this text based on the following guidelines: Positive: if a text implies positive sentiment, attitude, and emotional state. Negative: if a text implies negative sentiment or emotion. Neutral: if a text does not imply positive or negative language directly or indirectly. Provide sentiment labels only"

In [13]:
print("# Loading datasets")
sentiment_train_df = pd.read_parquet("hf://datasets/lelapa/SentimentTrain/data/train-00000-of-00001.parquet")

hau_dataset = Dataset.from_pandas(
    sentiment_train_df[sentiment_train_df['langs']=='hausa']
)
swa_dataset = Dataset.from_pandas(
    sentiment_train_df[sentiment_train_df['langs']=='swahili']
)

# If you need a DatasetDict to mimic the Hugging Face structure
dataset_dict = DatasetDict({
    "swahili": swa_dataset,
    "hausa": hau_dataset
})


# for swahili
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    max_new_tokens=15,
    task_instruction=sent_instruction,
    dataset=swa_dataset,
    csv_file_path=os.path.join(
        output_path,
        "swa_sent_prediction_dev.csv"
    ),
    custom_instruct=False,
)



# Loading datasets


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [14]:
# for hausa
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    task_instruction=sent_instruction,
    dataset=hau_dataset,
    csv_file_path=os.path.join(
        output_path,
        "hau_sent_prediction_dev.csv"
    ),
    max_new_tokens=15,
    custom_instruct=False,
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


MT baseline

In [15]:
# Load the Swahili dataset from a local CSV file
mt_train = pd.read_parquet("hf://datasets/lelapa/MTTrain/data/train-00000-of-00001.parquet")


hau_dataset = Dataset.from_pandas(
    mt_train[mt_train['langs']=='eng-hau']
)
swa_dataset = Dataset.from_pandas(
    mt_train[mt_train['langs']=='eng-swa']
)

# If you need a DatasetDict to mimic the Hugging Face structure
dataset_dict = DatasetDict({
    "swahili": swa_dataset,
    "hausa": hau_dataset
})

# Print to verify
print(swa_dataset)
print(hau_dataset)

Dataset({
    features: ['ID', 'task', 'langs', 'data_source', 'instruction', 'inputs', 'targets', '__index_level_0__'],
    num_rows: 300
})
Dataset({
    features: ['ID', 'task', 'langs', 'data_source', 'instruction', 'inputs', 'targets', '__index_level_0__'],
    num_rows: 300
})


In [16]:
# don't change this instruction
mt_instruction = "Translate the following from {input_lang} into {output_lang}."

In [17]:
hau_mt_instruction = mt_instruction.format(input_lang="English", output_lang="Hausa")
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    task_instruction=hau_mt_instruction,
    dataset=hau_dataset,
    csv_file_path=os.path.join(
        output_path,
        "hau_mt_prediction_dev.csv"
    ),
    custom_instruct=False,
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [18]:
swa_mt_instruction = mt_instruction.format(input_lang="English", output_lang="Swahili")
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    task_instruction=swa_mt_instruction,
    dataset=swa_dataset,
    csv_file_path=os.path.join(
        output_path,
        "swa_mt_prediction_dev.csv"
    ),
    custom_instruct=False,
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


XNLI baseline

In [58]:
xnli_train = pd.read_parquet("hf://datasets/lelapa/XNLITrain/data/train-00000-of-00001.parquet")
hau_dataset = Dataset.from_pandas(
    xnli_train[xnli_train['langs']=='hau']
)
swa_dataset = Dataset.from_pandas(
    xnli_train[xnli_train['langs']=='swa']
)

# If you need a DatasetDict to mimic the Hugging Face structure
dataset_dict = DatasetDict({
    "swahili": swa_dataset,
    "hausa": hau_dataset
})

# Print to verify
print(swa_dataset)
print(hau_dataset)

Dataset({
    features: ['ID', 'langs', 'premise', 'inputs', 'instruction', 'targets', '__index_level_0__'],
    num_rows: 200
})
Dataset({
    features: ['ID', 'langs', 'premise', 'inputs', 'instruction', 'targets', '__index_level_0__'],
    num_rows: 200
})


In [59]:
xnli_instruction = "Is the following question True, False or Neither?"

In [60]:
# for hausa
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    task_instruction=xnli_instruction,
    dataset=hau_dataset,
    csv_file_path=os.path.join(
        output_path,
        "hau_xnli_prediction_dev.csv"
    ),
    max_new_tokens=15,
    custom_instruct=False,
)

# for swahili
model_function.main(
    model,
    tokenizer,
    BASE_PROMPT,
    sample_size=3,
    task_instruction=xnli_instruction,
    dataset=swa_dataset,
    csv_file_path=os.path.join(
        output_path,
        "swa_xnli_prediction_dev.csv"
    ),
    max_new_tokens=15,
    custom_instruct=False,
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [68]:
from collections import Counter
def process_likelihood(likelihood_str: str) -> List[float]:
    """
    Process a likelihood string to clean and convert it to a list of floats.
    """
    # Clean the string to remove unwanted characters
    clean_str = (
        likelihood_str.replace("tensor(", "").replace(")", "").strip()
        .replace("[[", "").replace("]]", "").strip()
        .replace(" device='cuda:0'", "").replace(" dtype=torch.float16", "").strip()
        .replace("tensor", "").strip()
    )

    # Remove any empty strings caused by extra commas
    clean_str = clean_str.replace(",,", ",")  # Remove duplicate commas if they exist

    # Convert to a list of floats
    likelihood = [
        float(x) for x in clean_str.split(",") if x.strip()
    ]  # Ensure non-empty strings are converted
    return likelihood

def create_submission(output_path, test_flag: bool):
    """
    Creates submission files based on the provided test_flag.

    Args:
    test_flag (bool): If True, creates a test submission file; otherwise, creates a final submission file.
    """
    if test_flag:
        try:
            df1 = pd.read_csv(os.path.join(
                output_path,
                "hau_sent_prediction_dev.csv")
                 )
            df2 = pd.read_csv(os.path.join(
                output_path,
                "swa_sent_prediction_dev.csv")
            )
            df3 = pd.read_csv(os.path.join(
                output_path,
                "hau_mt_prediction_dev.csv")
                             )
            df4 = pd.read_csv(os.path.join(
                output_path,
                "swa_mt_prediction_dev.csv"))
            df5 = pd.read_csv(os.path.join(
                output_path,
                "hau_xnli_prediction_dev.csv"))
            df6 = pd.read_csv(os.path.join(
                output_path,
                "swa_xnli_prediction_dev.csv"))
        except FileNotFoundError as e:
            print(
                "Seems you have not completed all the tasks, please complete all the tasks before attempting to create your submission file"
            )
            raise e
    else:
        filename = "submission.csv"
        try:
            df1 = pd.read_csv(os.path.join(
                output_path,
                "hau_sent_prediction.csv"))
            df2 = pd.read_csv(os.path.join(
                output_path,
                "swa_sent_prediction.csv"))
            df3 = pd.read_csv(os.path.join(
                output_path,
                "hau_mt_prediction.csv"))
            df4 = pd.read_csv(os.path.join(
                output_path,
                "swa_mt_prediction.csv"))
            df5 = pd.read_csv(os.path.join(
                output_path,
                "hau_xnli_prediction.csv"))
            df6 = pd.read_csv(os.path.join(
                output_path,
                "swa_xnli_prediction.csv"))
        except FileNotFoundError as e:
            print(
                "Seems you have not completed all the tasks, please complete all the tasks before attempting to create your submission file"
            )
            raise e

    # Combine and process data
    resmt = pd.concat([df3, df4], ignore_index=True)
    res_log = pd.concat([df1, df2, df5, df6], ignore_index=True)
    res_log.drop(columns=["Response"], inplace=True)
    res_log.rename(columns={"Log-Likelihood": "Response"}, inplace=True)
    res = pd.concat([res_log, resmt], ignore_index=True)

    def process_row(row):
        if "xnli" in row["ID"] or "sent" in row["ID"]:
            likelihoods = process_likelihood(row["Response"])
            predicted_label = np.argmax(likelihoods)
            return predicted_label
        return row["Response"]  # Default for other cases

    # Update the Response column in-place
    res["Response"] = res.apply(process_row, axis=1)

    if test_flag:
        filename = os.path.join(
                output_path,
                "submission_test.csv")
        # Save the submission file
        submission = res[["ID", "Response", "Targets"]]
        submission.to_csv(filename, index=False)
    else:
        filename = os.path.join(
                output_path,
                "submission.csv")
        # Save the submission file
        submission = res[["ID", "Response"]]
        submission.to_csv(filename, index=False)
    return submission

def evaluate_zindi_by_language(df):
    from collections import defaultdict, Counter
    import numpy as np

    results = defaultdict(lambda: defaultdict(list))  # results[task][lang] = [(true, pred), ...]
    chrfs_scores = defaultdict(list)

    for _, row in df.iterrows():
        ID = row["ID"].lower()

        # --- Task detection ---
        if "sentiment" in ID or "sent" in ID:
            task = "sent"
        elif "xnli" in ID or "afrixnli" in ID:
            task = "xnli"
        elif "mt" in ID:
            task = "mt"
        else:
            continue  # skip unknown tasks

        # --- Language detection ---
        if any(x in ID for x in ["swahili", "swa", "eng-swa"]):
            lang = "swahili"
        elif any(x in ID for x in ["hausa", "hau", "eng-hau"]):
            lang = "hausa"
        else:
            lang = "unknown"

        # --- Scoring logic ---
        if task == "sent":
            labels = {
                "swahili": ["Chanya", "Wastani", "Hasi"],
                "hausa": ["Kyakkyawa", "Tsaka-tsaki", "Korau"]
            }.get(lang, [])
            if not labels:
                continue
            label_to_id = {label.lower(): i for i, label in enumerate(labels)}
            true = label_to_id.get(row["Targets"].strip().lower(), -1)
            pred = int(row["Response"])
            if true != -1:
                results[task][lang].append((true, pred))

        elif task == "xnli":
            try:
                true = int(row["Targets"])
                pred = int(row["Response"])
                results[task][lang].append((true, pred))
            except:
                continue  # skip rows with invalid labels

        elif task == "mt":
            ref = row["Targets"]
            hyp = row["Response"]
            chrfs = chrF(reference=ref, hypothesis=hyp)
            chrfs_scores[lang].append(chrfs)

    # --- Aggregation ---
    report = {}
    for task in results:
        for lang in results[task]:
            y_true, y_pred = zip(*results[task][lang])
            f1 = calculate_f1(np.array(y_true), np.array(y_pred), num_classes=3)
            report[f"{task}_{lang}"] = round(f1, 4)

    for lang in chrfs_scores:
        avg_chrfs = np.mean(chrfs_scores[lang])
        report[f"mt_{lang}"] = round(avg_chrfs, 4)

    # --- Zindi score ---
    zindi_score = np.mean(list(report.values()))
    report["zindi_score"] = round(zindi_score, 4)

    return report


# From scratch implementation of chrf
def get_char_ngrams(sentence, n):
    """Generate character n-grams from a sentence."""
    sentence = sentence.replace(" ", "")  # Remove spaces for chrF
    return [sentence[i : i + n] for i in range(len(sentence) - n + 1)]


def precision_recall(reference, hypothesis, n):
    """Calculate precision and recall for character n-grams."""
    ref_ngrams = get_char_ngrams(reference, n)
    hyp_ngrams = get_char_ngrams(hypothesis, n)

    ref_count = Counter(ref_ngrams)
    hyp_count = Counter(hyp_ngrams)

    common_ngrams = ref_count & hyp_count
    true_positives = sum(common_ngrams.values())

    precision = true_positives / max(len(hyp_ngrams), 1)
    recall = true_positives / max(len(ref_ngrams), 1)

    return precision, recall


def f_score(precision, recall, beta=1):
    """Calculate the F1 score."""
    if precision + recall == 0:
        return 0.0
    return (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall)


def chrF(reference, hypothesis, max_n=6, beta=2):
    """Calculate the chrF score from scratch."""
    precisions = []
    recalls = []

    for n in range(1, max_n + 1):
        precision, recall = precision_recall(reference, hypothesis, n)
        precisions.append(precision)
        recalls.append(recall)

    avg_precision = sum(precisions) / max_n
    avg_recall = sum(recalls) / max_n

    return f_score(avg_precision, avg_recall, beta)


# From scratch implementation f1score 3 class
def calculate_f1(true_labels, pred_labels, num_classes):
    f1_scores = []

    for i in range(num_classes):
        TP = np.sum((true_labels == i) & (pred_labels == i))  # True Positives
        FP = np.sum((true_labels != i) & (pred_labels == i))  # False Positives
        FN = np.sum((true_labels == i) & (pred_labels != i))  # False Negatives

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0

        # Calculate F1 score
        f1 = (
            2 * (precision * recall) / (precision + recall)
            if (precision + recall) > 0
            else 0
        )
        f1_scores.append(f1)

    macro_f1 = np.mean(f1_scores)

    return macro_f1


In [62]:
score_test = create_submission(output_path=output_path, test_flag=True)

In [63]:
score_test

,ID,Response,Targets
0,ID_6aba33a1_sentiment_ dev_hausa,1,Kyakkyawa
1,ID_ce64d307_sentiment_ dev_hausa,1,Tsaka-tsaki
2,ID_2efc9515_sentiment_ dev_hausa,1,Tsaka-tsaki
3,ID_dfb02831_sentiment_ dev_swahili,1,Wastani
4,ID_ad1d9888_sentiment_ dev_swahili,1,Wastani
5,ID_34eed91d_sentiment_ dev_swahili,1,Wastani
6,ID_648d37ff_dev_afrixnli_hau,1,0
7,ID_f96a39cb_dev_afrixnli_hau,1,0
8,ID_99a61e3d_dev_afrixnli_hau,1,2
9,ID_4c5d953b_dev_afrixnli_swa,1,1


In [67]:
print(score_test["ID"].unique().tolist())

['ID_6aba33a1_sentiment_ dev_hausa', 'ID_ce64d307_sentiment_ dev_hausa', 'ID_2efc9515_sentiment_ dev_hausa', 'ID_dfb02831_sentiment_ dev_swahili', 'ID_ad1d9888_sentiment_ dev_swahili', 'ID_34eed91d_sentiment_ dev_swahili', 'ID_648d37ff_dev_afrixnli_hau', 'ID_f96a39cb_dev_afrixnli_hau', 'ID_99a61e3d_dev_afrixnli_hau', 'ID_4c5d953b_dev_afrixnli_swa', 'ID_a38aa9d8_dev_afrixnli_swa', 'ID_643fa526_dev_afrixnli_swa', 'ID_5e3d2251_dev_mt_eng-hau', 'ID_fd263d98_dev_mt_eng-hau', 'ID_96a1f4d4_dev_mt_eng-hau', 'ID_4185b417_dev_mt_eng-swa', 'ID_a694c64e_dev_mt_eng-swa', 'ID_13e437e9_dev_mt_eng-swa']


In [69]:
score_report = evaluate_zindi_by_language(score_test)
score_report

{'sent_hausa': np.float64(0.2667),
 'sent_swahili': np.float64(0.3333),
 'xnli_hausa': np.float64(0.0),
 'xnli_swahili': np.float64(0.3333),
 'mt_hausa': np.float64(0.0),
 'mt_swahili': np.float64(0.1614),
 'zindi_score': np.float64(0.1824)}

In [72]:
# 1. How many rows of xnli_hausa?
df = score_test
df[(df["ID"].str.contains("xnli", case=False) | df["ID"].str.contains("afrixnli", case=False)) &
   (df["ID"].str.contains("hau", case=False))]


,ID,Response,Targets
6,ID_648d37ff_dev_afrixnli_hau,1,0
7,ID_f96a39cb_dev_afrixnli_hau,1,0
8,ID_99a61e3d_dev_afrixnli_hau,1,2
